In [ ]:
# pip install requests beautifulsoup4

In [2]:
import re
import sys
import time
import sqlite3
from pathlib import Path
from difflib import SequenceMatcher
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

DB_PATH = Path("data/db.sqlite")
OUT_DIR = Path("data/supplier_products")
CATALOG_FILE = Path("supplier_product_list")
DELAY = 1.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# Maps domain keywords to DB supplier names
DOMAIN_TO_SUPPLIER: dict[str, str] = {
    "bulksupplements":  "BulkSupplements",
    "capsuline":        "Capsuline",
    "customprobiotics": "Custom Probiotics",
    "feedsforless":     "Nutra Blend",
    "purebulk":         "PureBulk",
    "source-omega":     "Source-Omega LLC",
    "spectrumchemical": "Spectrum Chemical",
    "traceminerals":    "Trace Minerals",
}

In [3]:
def load_catalog_urls() -> dict[str, str]:
    """Returns {supplier_name: catalog_url} by matching supplier_product_list domains."""
    result: dict[str, str] = {}
    for line in CATALOG_FILE.read_text().splitlines():
        url = line.strip()
        if not url:
            continue
        host = urlparse(url).netloc.lower()
        for keyword, supplier in DOMAIN_TO_SUPPLIER.items():
            if keyword in host:
                result[supplier] = url
                break
    return result


def get_db_products() -> dict[str, list[dict]]:
    """Returns {supplier_name: [{SupplierId, SupplierName, ProductId, SKU}]}"""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("""
        SELECT sp.SupplierId, s.Name AS SupplierName,
               sp.ProductId, p.SKU
        FROM Supplier_Product sp
        JOIN Supplier s ON s.Id = sp.SupplierId
        JOIN Product p  ON p.Id = sp.ProductId
        ORDER BY s.Name, p.SKU
    """).fetchall()
    conn.close()
    by_supplier: dict[str, list[dict]] = {}
    for r in rows:
        d = dict(r)
        by_supplier.setdefault(d["SupplierName"], []).append(d)
    return by_supplier


catalog_urls = load_catalog_urls()
all_products = get_db_products()

print("Catalog URLs:", list(catalog_urls.keys()))
print("Suppliers in DB with products:", len(all_products))

Catalog URLs: ['BulkSupplements', 'Capsuline', 'Custom Probiotics', 'Nutra Blend', 'PureBulk', 'Source-Omega LLC', 'Spectrum Chemical', 'Trace Minerals']
Suppliers in DB with products: 40


In [4]:
def sku_to_ingredient(sku: str) -> str:
    """RM-C2-ascorbic-acid-4823fabf → 'ascorbic acid'"""
    m = re.match(r"^(?:RM|FG)-C\d+-(.+)-[0-9a-f]{8}$", sku)
    if m:
        return m.group(1).replace("-", " ").lower()
    return sku.lower()


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def best_match(page_title: str, candidates: list[dict], threshold: float = 0.45) -> dict | None:
    best, best_score = None, threshold
    title_norm = page_title.lower()
    for p in candidates:
        ingredient = sku_to_ingredient(p["SKU"])
        score = similarity(ingredient, title_norm)
        if ingredient in title_norm or title_norm in ingredient:
            score = max(score, 0.7)
        if score > best_score:
            best_score = score
            best = p
    return best


def safe_filename(s: str) -> str:
    return re.sub(r"[^\w\-]", "_", s)


def fetch(url: str, session: requests.Session) -> requests.Response | None:
    try:
        r = session.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        return r
    except Exception as e:
        print(f"  WARN: {url} → {e}")
        return None


def get_page_title(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(strip=True)
    title = soup.find("title")
    if title:
        return title.get_text(strip=True).split("|")[0].split("\u2013")[0].strip()
    return ""

In [5]:
def shopify_product_urls(catalog_url: str, session: requests.Session) -> list[str]:
    """Enumerate all product URLs via Shopify's collection JSON API."""
    parsed = urlparse(catalog_url)
    base = f"{parsed.scheme}://{parsed.netloc}"
    parts = [p for p in parsed.path.strip("/").split("/") if p]
    if "collections" in parts:
        idx = parts.index("collections")
        collection = parts[idx + 1] if idx + 1 < len(parts) else "all"
    else:
        collection = "all"

    urls: list[str] = []
    page = 1
    while True:
        api = f"{base}/collections/{collection}/products.json?limit=250&page={page}"
        resp = fetch(api, session)
        if not resp:
            break
        products = resp.json().get("products", [])
        if not products:
            break
        for p in products:
            handle = p.get("handle", "")
            if handle:
                urls.append(f"{base}/products/{handle}")
        page += 1
        time.sleep(DELAY)
    return urls


def generic_product_urls(catalog_url: str, session: requests.Session) -> list[str]:
    """Scrape product links from a catalog page via heuristic link extraction."""
    resp = fetch(catalog_url, session)
    if not resp:
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    parsed = urlparse(catalog_url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    skip = ("/cart", "/checkout", "/account", "/login",
            "/search", "/blog", "/news", "/contact", "/about",
            "/policy", "/faq")
    seen: set[str] = set()
    links: list[str] = []

    for a in soup.find_all("a", href=True):
        href = a["href"].split("?")[0].split("#")[0]
        full = urljoin(base, href)
        fp = urlparse(full)
        if fp.netloc != parsed.netloc:
            continue
        path = fp.path.rstrip("/")
        if any(path.startswith(s) for s in skip):
            continue
        if full in seen or full == catalog_url or path == parsed.path:
            continue
        segments = [s for s in path.split("/") if s]
        if len(segments) >= 2:
            seen.add(full)
            links.append(full)

    return links


def is_shopify(catalog_url: str) -> bool:
    return "/collections/" in catalog_url

In [6]:
def scrape_supplier(
    supplier_name: str,
    catalog_url: str,
    products: list[dict],
    session: requests.Session,
) -> None:
    print(f"\n{'='*60}")
    print(f"Supplier : {supplier_name}  ({len(products)} products in DB)")
    print(f"Catalog  : {catalog_url}")

    print("  Collecting product URLs...")
    if is_shopify(catalog_url):
        product_urls = shopify_product_urls(catalog_url, session)
    else:
        product_urls = generic_product_urls(catalog_url, session)
    print(f"  Found {len(product_urls)} product URLs on catalog")

    # Allow resuming: skip products already saved to disk
    sid_prefix = str(products[0]["SupplierId"])
    saved_pids = {
        int(f.name.split("_")[2])
        for f in OUT_DIR.glob(f"{sid_prefix}_*_*_*.html")
    }
    remaining = [p for p in products if p["ProductId"] not in saved_pids]

    matched_ids: set[int] = set()
    saved = 0

    for url in product_urls:
        time.sleep(DELAY)
        resp = fetch(url, session)
        if not resp:
            continue

        soup = BeautifulSoup(resp.text, "html.parser")
        title = get_page_title(soup)
        if not title:
            continue

        candidates = [p for p in remaining if p["ProductId"] not in matched_ids]
        if not candidates:
            break

        match = best_match(title, candidates)
        if not match:
            continue

        matched_ids.add(match["ProductId"])
        sname = safe_filename(match["SupplierName"])
        pid   = match["ProductId"]
        psku  = match["SKU"]
        fname = f"{sid_prefix}_{sname}_{pid}_{psku}.html"
        (OUT_DIR / fname).write_text(resp.text, encoding="utf-8")
        print(f"  SAVED  : {fname}  ← '{title}'")
        saved += 1

    unmatched = [p for p in remaining if p["ProductId"] not in matched_ids]
    print(f"\n  Saved: {saved}  |  Unmatched DB products: {len(unmatched)}")
    if unmatched:
        print("  Unmatched SKUs (first 10):")
        for p in unmatched[:10]:
            print(f"    {p['SKU']}")

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
session = requests.Session()

for supplier_name, catalog_url in catalog_urls.items():
    products = all_products.get(supplier_name, [])
    if not products:
        print(f"Skipping {supplier_name}: no Supplier_Product entries in DB")
        continue
    scrape_supplier(supplier_name, catalog_url, products, session)


Supplier : BulkSupplements  (6 products in DB)
Catalog  : https://www.bulksupplements.com/collections/new-products?page=1
  Found 78 product URLs on catalog

  Saved: 0  |  Unmatched DB products: 6
  Unmatched SKUs (first 10):
    RM-C18-bcaas-162e8977
    RM-C40-l-isoleucine-fc3f6810
    RM-C40-l-leucine-dc251fa2
    RM-C40-l-valine-237d9405
    RM-C53-leucine-d14d0c88
    RM-C55-taurine-e1cf4b6c

Supplier : Capsuline  (26 products in DB)
Catalog  : https://eu.capsuline.com/collections/bulk
  Found 36 product URLs on catalog
  SAVED  : 9_Capsuline_549_RM-C30-vegetarian-capsule-ea92a1d1.html  ← 'Clear Vegetarian Capsules Size 00 (Box of 75,000)'
  SAVED  : 9_Capsuline_259_RM-C11-gelatin-capsule-bovine-42ef5483.html  ← 'Clear Vegetarian Capsules Size 0 (Box of 100,000)'
  SAVED  : 9_Capsuline_292_RM-C14-gelatin-017eb28d.html  ← 'Clear Gelatin Capsules Size 0 (Box of 100,000)'
  SAVED  : 9_Capsuline_357_RM-C17-gelatin-45b7cd2c.html  ← 'Colored Empty Gelatin Capsules Size 00 (Box of 75,0